# Lab 11: Implementing Scaled Dot-Product & Multi-Head Self-Attention from Scratch

Welcome to Laboratory 11! In this lab, we build the core computational engine behind Modern AI and Large Language Models: the **Transformer Self-Attention Mechanism** (*Vaswani et al., "Attention Is All You Need"*):
1. **Scaled Dot-Product Attention**: Implement the foundational $\text{Softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$ operator with causal/padding masking.
2. **Multi-Head Self-Attention (`MultiHeadAttention`)**: Split embeddings into multiple subspace projection heads for multi-aspect semantic feature routing.
3. **Transformer Encoder Block**: Assemble Layer Normalization, Residual Connections, and Feed-Forward Networks (FFN).


## 1. Technical Preliminaries & Imports


In [ ]:
# Import PyTorch and mathematical utility modules
import torch
import torch.nn as nn
import math

# Seed for reproducibility
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Device:', device)


## 2. Scaled Dot-Product Attention from First Principles

### Mathematical Formulation
Given Queries $\mathbf{Q}$, Keys $\mathbf{K}$, and Values $\mathbf{V}$ with projection dimension $d_k$:
$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left( \frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}} + \mathbf{M} \right) \mathbf{V}$$

* **Scaling Factor $\frac{1}{\sqrt{d_k}}$**: For large $d_k$, dot products grow large in magnitude, pushing softmax into regions with vanishingly small gradients. Scaling by $\sqrt{d_k}$ preserves unit variance.
* **Mask $\mathbf{M}$**: Adds $-\infty$ (or $-10^9$) to prevent attending to future tokens (causal masking) or padding tokens.


### Helper Function: `scaled_dot_product_attention`
The function below computes scaled dot-product attention scores and output representations.


In [ ]:
def scaled_dot_product_attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, mask: torch.Tensor = None):
    """Computes Scaled Dot-Product Attention with optional masking.
    
    Args:
        Q: Query tensor of shape (..., Seq_Len_Q, d_k)
        K: Key tensor of shape (..., Seq_Len_K, d_k)
        V: Value tensor of shape (..., Seq_Len_V, d_v)
        mask: Optional boolean tensor of shape (..., Seq_Len_Q, Seq_Len_K)
    Returns:
        output: Contextualized values tensor of shape (..., Seq_Len_Q, d_v)
        attn_weights: Normalized attention probability matrix of shape (..., Seq_Len_Q, Seq_Len_K)
    """
    # Step 1: Extract feature dimensionality d_k from queries
    d_k = Q.size(-1)
    
    # Step 2: Compute pairwise query-key compatibility scores: Q @ K^T / sqrt(d_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Step 3: Apply mask (replace masked positions with large negative value before softmax)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
        
    # Step 4: Softmax normalization along key dimension to obtain attention probability distribution
    attn_weights = torch.softmax(scores, dim=-1)
    
    # Step 5: Compute weighted sum of values: Attention_Weights @ V
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

# Verify Scaled Dot-Product Attention with dummy tensors
batch_size, num_heads, seq_len, head_dim = 1, 4, 8, 16
q = torch.randn(batch_size, num_heads, seq_len, head_dim)
k = torch.randn(batch_size, num_heads, seq_len, head_dim)
v = torch.randn(batch_size, num_heads, seq_len, head_dim)

attn_out, weights = scaled_dot_product_attention(q, k, v)
print('Attention Output Tensor Shape:  ', attn_out.shape)
print('Attention Weights Matrix Shape: ', weights.shape)
assert attn_out.shape == (1, 4, 8, 16), 'Attention output shape mismatch!'
print('[Verification Passed] Scaled Dot-Product Attention operates correctly!')


## 3. Multi-Head Self-Attention Architecture

### Architecture Overview: `MultiHeadAttention`
Multi-Head Attention projects input representations into $h$ different subspaces in parallel:
1. **Linear Projections**: Projects $X \to Q, K, V$ using $W_q, W_k, W_v \in \mathbb{R}^{d_{model} \times d_{model}}$.
2. **Head Splitting & Transpose**: Reshapes tensors from `(B, T, d_model)` to `(B, h, T, d_k)` where $d_k = d_{model} / h$.
3. **Parallel Attention**: Computes scaled dot-product attention simultaneously across all $h$ heads.
4. **Concatenation & Output Linear Projection**: Concatenates head outputs back to `(B, T, d_model)` and passes through linear projection $W_o$.


In [ ]:
# Define the Complete Multi-Head Attention Architecture
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention module with parallel projection subspaces."""
    def __init__(self, d_model: int = 64, num_heads: int = 4):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads # Dimension per individual attention head
        
        # Learnable linear projections for Query, Key, Value, and Final Output
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x: Input sequence tensor of shape (Batch_Size, Seq_Len, d_model)
            mask: Optional attention mask
        Returns:
            Projected attention output of shape (Batch_Size, Seq_Len, d_model)
        """
        B, seq_len, _ = x.shape
        
        # 1. Project and reshape to (B, num_heads, Seq_Len, d_k)
        Q = self.W_q(x).view(B, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # 2. Compute Scaled Dot-Product Attention in parallel across all heads
        attn_out, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
        
        # 3. Transpose and concatenate heads: (B, Seq_Len, num_heads * d_k) = (B, Seq_Len, d_model)
        concat_out = attn_out.transpose(1, 2).contiguous().view(B, seq_len, self.d_model)
        
        # 4. Final output projection
        return self.W_o(concat_out)

# Instantiate and verify MultiHeadAttention
mha_module = MultiHeadAttention(d_model=64, num_heads=4).to(device)
dummy_input = torch.randn(2, 10, 64).to(device) # Batch=2, Seq_Len=10, d_model=64
mha_output = mha_module(dummy_input)

print('Input Sequence Tensor Shape:  ', dummy_input.shape)
print('MHA Output Tensor Shape:      ', mha_output.shape)
assert mha_output.shape == (2, 10, 64), 'MHA output shape mismatch!'
print('[Verification Passed] Multi-Head Attention executed successfully!')


## 4. Summary & Takeaways
1. **Direct Pairwise Connectivity**: Attention connects every token to every other token in $\mathcal{O}(1)$ path length, overcoming RNN vanishing gradient bottlenecks.
2. **Multi-Head Subspaces**: Splitting into multiple heads enables the model to simultaneously attend to syntactic relations, semantic associations, and positional context.
3. **Scaling Factor**: $\frac{1}{\sqrt{d_k}}$ ensures numerically stable softmax gradients during backpropagation.
